# 9주차 (1/2). 이상진단 전용 모델 — 점수 생성 (GPU 서버)

In [ ]:
import sys, json, time, platform, subprocess, traceback
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))
from src import anomaly_adapter as aa
aa.setup()  # np.Inf 별칭 복원 + TSB-AD·CATCH·DADA 경로 등록

OUT = ROOT / "results/wk9"
SC = OUT / "scores"
SC.mkdir(parents=True, exist_ok=True)
RUN_LOG, FAIL_LOG = OUT / "run_log.csv", OUT / "failures.csv"

try:
    import torch
    GPU = torch.cuda.is_available()
except ImportError:
    torch, GPU = None, False

commits = {}
for name, want in aa.COMMITS.items():
    commits[name] = subprocess.run(["git", "-C", str(aa.REPOS[name]), "rev-parse", "--short", "HEAD"],
                                   capture_output=True, text=True).stdout.strip()
    if not want.startswith(commits[name][:7]):
        print(f"⚠ {name} 커밋 {commits[name]} — requirements_stage2.txt 핀은 {want}")

env = {"python": platform.python_version(), "torch": getattr(torch, "__version__", None),
       "gpu": torch.cuda.get_device_name(0) if GPU else None, "commits": commits,
       "catch_cfg": aa.CATCH_CFG, "dada_win": aa.DADA_WIN, "seed": 2021}
(OUT / "env.json").write_text(json.dumps(env, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(env, ensure_ascii=False, indent=1))
print("GPU 사용 가능" if GPU else "⚠ GPU 없음 — CATCH·DADA 셀은 건너뜁니다(고전 3종·LightGBM 잔차는 CPU로 가능)")

def log_run(method, building, secs, **kw):
    row = pd.DataFrame([{"method": method, "building": building, "secs": secs, **kw}])
    row.to_csv(RUN_LOG, mode="a", header=not RUN_LOG.exists(), index=False, encoding="utf-8")

def log_fail(method, building, err):
    row = pd.DataFrame([{"method": method, "building": building, "error": repr(err)[:300]}])
    row.to_csv(FAIL_LOG, mode="a", header=not FAIL_LOG.exists(), index=False, encoding="utf-8")

## 과제 9-1. TSB-AD 평가 모듈 도입 + LEAD 1.0 → TSB-AD 형식

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from TSB_AD.utils.slidingWindows import find_length_rank

X, y = aa.load_2016()
print("X", X.shape, X.index[0], "→", X.index[-1], "| 라벨 매칭 건물:", y.shape[1], "/ 17")
assert y.shape[1] == 17 and X.index.is_monotonic_increasing

# TSB-AD 형식 파일 (Data, Label) — results/wk9/tsb_format/
FMT = OUT / "tsb_format"
FMT.mkdir(exist_ok=True)
for c in X.columns:
    pd.DataFrame({"Data": X[c].to_numpy(), "Label": y[c].astype(int).to_numpy()}).to_csv(FMT / f"{c}.csv", index=False, encoding="utf-8")

WIN = {c: int(find_length_rank(MinMaxScaler().fit_transform(X[[c]]), rank=1)) for c in X.columns}
ev = {c: aa.events(y[c]) for c in X.columns}
prof = pd.DataFrame({
    "이상 시점": y.sum(), "이상률(%)": y.mean() * 100, "이상 구간 수": {c: len(ev[c]) for c in ev},
    "중앙 구간길이(h)": {c: ev[c].length.median() if len(ev[c]) else np.nan for c in ev},
    "최장 구간(h)": {c: ev[c].length.max() if len(ev[c]) else np.nan for c in ev},
    "slidingWindow": pd.Series(WIN),
})
prof.to_csv(OUT / "profile.csv", encoding="utf-8")
print(prof.round(2).to_string())
print("\n평가 가능 건물(이상 ≥ 1):", len(aa.evaluable(y)), "/ 17 | 전체 이상률 %.2f%%" % (y.to_numpy().mean() * 100))

# 평가 모듈 점검: 라벨 그대로를 점수로 주면 VUS-PR ≈ 1, 무작위 점수면 ≈ 이상률
c = max(aa.evaluable(y), key=lambda k: y[k].sum())
perfect = aa.evaluate(y[c].astype(float), y[c], WIN[c])
rand = aa.evaluate(np.random.default_rng(0).random(len(y)), y[c], WIN[c])
print(f"\n[{c}] 완벽한 점수 VUS-PR={perfect['VUS-PR']:.3f} | 무작위 점수 VUS-PR={rand['VUS-PR']:.3f} (이상률 {y[c].mean():.3f})")

## 과제 9-3 (1/3). 고전 3종 + LightGBM 잔차 — CPU

In [ ]:
from joblib import Parallel, delayed

N_JOBS = 4  # 서버 코어 수에 맞춰 조정
def one(c):
    t0 = time.perf_counter()
    s, w, secs = aa.classic_scores(X[c].to_numpy())
    return c, s, w, secs

if not all((SC / f"{m}.parquet").exists() for m in ["Sub-LOF", "MatrixProfile", "IsolationForest"]):
    res = Parallel(n_jobs=N_JOBS)(delayed(one)(c) for c in X.columns)
    for m in ["Sub-LOF", "MatrixProfile", "IsolationForest"]:
        pd.DataFrame({c: s[m] for c, s, _, _ in res}, index=X.index).to_parquet(SC / f"{m}.parquet")
        for c, _, _, secs in res:
            log_run(m, c, secs[m], device="CPU", n_jobs=1)
    assert all(w == WIN[c] for c, _, w, _ in res), "래퍼 내부 창과 WIN 불일치"

lg = aa.lgbm_residual_scores(X.index, X.columns)
lg.to_parquet(SC / "LightGBM잔차.parquet")
print("저장된 점수:", sorted(p.stem for p in SC.glob("*.parquet")))

## 과제 9-3 (2/3). CATCH — GPU 학습

In [ ]:
CATCH_PQ = SC / "CATCH.parquet"
if not GPU:
    print("GPU 없음 — 건너뜀")
elif CATCH_PQ.exists():
    print("이미 있음:", CATCH_PQ.name)
else:
    try:
        torch.cuda.reset_peak_memory_stats()
        score, fit_s = aa.catch_scores(X, seed=2021)
        assert score.shape == X.shape and np.isfinite(score.to_numpy()).all()
        score.to_parquet(CATCH_PQ)
        log_run("CATCH", "all17", fit_s, device="GPU", peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9)
        print("저장:", CATCH_PQ.name, "| 학습 %.1fs | VRAM %.2fGB" % (fit_s, torch.cuda.max_memory_allocated() / 1e9))
    except Exception as e:
        log_fail("CATCH", "all17", e)
        traceback.print_exc()
        print("30분 룰: 원인을 failures.csv·보고서 6번에 기록하고 넘어간다")

## 과제 9-3 (3/3). DADA — GPU 추론

In [ ]:
DADA_PQ = SC / "DADA.parquet"
if not GPU:
    print("GPU 없음 — 건너뜀")
elif DADA_PQ.exists():
    print("이미 있음:", DADA_PQ.name)
else:
    try:
        t0 = time.perf_counter()
        dada = aa.load_dada("cuda")
        print("DADA 로드 %.1fs | 파라미터 %.2fM" % (time.perf_counter() - t0, sum(p.numel() for p in dada.parameters()) / 1e6))
        cols = {}
        for c in X.columns:
            torch.cuda.synchronize(); t0 = time.perf_counter()
            cols[c] = aa.dada_scores(dada, X[c].to_numpy())
            torch.cuda.synchronize()
            log_run("DADA", c, time.perf_counter() - t0, device="GPU", peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9)
        pd.DataFrame(cols, index=X.index).to_parquet(DADA_PQ)
        print("저장:", DADA_PQ.name)
    except Exception as e:
        log_fail("DADA", "all", e)
        traceback.print_exc()
        print("30분 룰: 원인을 failures.csv·보고서 6번에 기록하고 넘어간다")

In [ ]:
# 요약: 점수 파일 · 시간 · 실패
sc = {p.stem: pd.read_parquet(p) for p in SC.glob("*.parquet")}
for m, d in sc.items():
    assert d.shape == X.shape and np.isfinite(d.to_numpy()).all(), f"{m}: 모양/NaN 점검 실패"
print("점수 파일:", {m: d.shape for m, d in sc.items()})
if RUN_LOG.exists():
    print(pd.read_csv(RUN_LOG).groupby(["method", "device"])["secs"].agg(["count", "sum", "mean"]).round(2))
if FAIL_LOG.exists():
    print("\n실패 기록:\n", pd.read_csv(FAIL_LOG).to_string(index=False))